# Behavioral Biomarker Training (HMOG)
This notebook implements anomaly detection for behavioural sensor data. It detects deviations in movement patterns that might indicate cognitive decline or physical distress.

### Cell 1: Feature Extraction Logic
We define `extract_behavioral_features` to calculate statistical moments (mean, std, magnitude) from raw accelerometer streams.

In [ ]:
%run ../data/Data_Management.ipynb
from sklearn.ensemble import IsolationForest
from tqdm.auto import tqdm
import joblib, os, pandas as pd, numpy as np
loader = DataLoader()
def extract_behavioral_features(df):
    df_c = clean_hmog_data(df)
    feat = {"acc_x_mean": df_c["acc_x"].mean(), "acc_y_mean": df_c["acc_y"].mean(), "acc_z_mean": df_c["acc_z"].mean(), "acc_mag_mean": np.sqrt(df_c["acc_x"]**2 + df_c["acc_y"]**2 + df_c["acc_z"]**2).mean()}
    return pd.Series(feat)

### Cell 2: Persistence and Engineering
Processing 6GB of sensor data is slow, so we cache the engineered features into a PKL file for instant reloading in future sessions.

In [ ]:
PATH = os.path.join(BASE_DATA_PATH, "HMOG", "processed_hmog_biomarkers.pkl")
if os.path.exists(PATH):
    features_df = pd.read_pickle(PATH)
else:
    all_f, users = [], [u for u in os.listdir(HMOG_LOCAL_PATH) if os.path.isdir(os.path.join(HMOG_LOCAL_PATH, u))]
    for u in tqdm(users):
        u_dir = os.path.join(HMOG_LOCAL_PATH, u, u)
        if not os.path.exists(u_dir): continue
        for s_dir in [d for d in os.listdir(u_dir) if "session" in d]:
            df_raw = loader.load_hmog_session(u, s_dir.split("_")[-1], "Accelerometer")
            if df_raw is not None:
                try: all_f.append(extract_behavioral_features(df_raw))
                except: continue
    features_df = pd.DataFrame(all_f)
    features_df.to_pickle(PATH)

### Cell 3: Anomaly Detection
We use an Isolation Forest with 5% contamination. This model learns "normal" patterns and flags everything else as an anomaly (-1).

In [ ]:
bms_model = IsolationForest(contamination=0.05, random_state=42)
bms_model.fit(features_df.values)
joblib.dump(bms_model, "../../models/behavioral_biomarker_v2.joblib")